# Mercari Price Analysis — Pandas & NumPy Reference Notebook
> Runs clean top-to-bottom. Every cell answers **why** it exists, not just what it does.

## Imports

In [ ]:
import pandas as pd  # aliased to pd so 200+ calls stay readable without the full module name cluttering every line
import numpy as np   # aliased to np; vectorized math on arrays runs at C speed — critical when looping over 50k rows would be ~100× slower

## Series — Building Intuition on Small Data Before Scaling Up

In [ ]:
prices = pd.Series(
    [2400, 45000, 150, 89000, 3500],
    index=["shoes", "phone", "book", "bag", "mouse"]
)  # named index so lookups read like human intent (prices["phone"]) instead of magic numbers that shift if rows are reordered

In [ ]:
prices["phone"]  # label access is unambiguous — integer position [1] would silently return the wrong item if the Series were re-sorted

In [ ]:
prices * 0.9  # vectorized multiply broadcasts the scalar across every element at C speed — a Python for-loop would be ~50× slower on large data

In [ ]:
prices[prices > 1000]  # boolean mask filters WITHOUT modifying the original Series — safe to reuse prices downstream unchanged

## DataFrame — Toy Example Before Real Data

In [ ]:
df = pd.DataFrame({
    "name":     ["Nike", "iPhone", "Book", "Gucci"],
    "price":    [2400, 45000, 150, 89000],
    "shipping": [1, 0, 1, 0]
})  # tiny 4-row df isolates concept from data-scale noise — mistakes here are obvious, not buried in 50k rows

In [ ]:
df["price"]  # single bracket returns a Series — needed whenever you pass a column to a function that expects 1D data

In [ ]:
df[["name", "price"]]  # double bracket returns a DataFrame — use when you need multiple columns together (e.g., to display a report)

In [ ]:
df.iloc[0]  # integer position 0 always means "first row" regardless of index value — safe for position-based iteration

---
## Load the Mercari Dataset

In [ ]:
# FIX: changed from absolute Windows path (D:\Career\...) to relative path
# so anyone who clones this repo can run it without editing this cell
df = pd.read_csv("mercari_sample.csv")

In [ ]:
df.shape  # checking dimensions FIRST reveals the memory cost upfront — (50000, 8) means ~3 MB, safe to hold fully in RAM

In [ ]:
df.head()  # sanity-check that CSV columns mapped correctly before spending any time on cleaning or analysis

In [ ]:
df.info()  # reveals dtypes AND null counts in one pass — tells us exactly which columns need cleaning before we waste time on analysis

In [ ]:
df.describe()  # stats expose obvious data quality problems immediately — price min=0 and max=1506 tells us there are free items AND outliers

In [ ]:
# BUG FIX: was df.isnull().count() which always returns len(df) for every column (counts non-nulls, not nulls)
# .sum() totals the True flags from .isnull(), giving the actual null count per column
df.isnull().sum()

In [ ]:
df.isnull().sum() / len(df) * 100  # converts raw null counts to percentages so we can apply the drop-vs-fill decision rule:
                                    # <5% → safe to drop | >5% → fill with a sentinel to preserve the row count

In [ ]:
df["price"].dtype  # confirming price is float64 so vectorized math (*, /, np.log) works without silent string coercion

## `iloc` & `loc` — Precise Row/Column Access

| | `iloc` | `loc` |
|---|---|---|
| Input type | **Integer position** | **Label** |
| Slice end | **Excluded** (like Python list) | **Included** |
| Use when | You know the row number | You know the row label or a condition |

### `iloc` — Integer Position (end **excluded**)

In [ ]:
df.iloc[0]  # integer 0 is unambiguous — safe even if the index was reset or non-sequential after filtering

In [ ]:
df.iloc[0:5]  # slice 0-4 (end is EXCLUDED like Python lists) — useful for a quick preview during EDA

In [ ]:
df.iloc[2, 1]  # [row, col] by position — pinpoints a single cell without knowing column name (handy in loops)

### `loc` — Label-Based (end **included**)

In [ ]:
df.loc[0:5]  # label 0 through 5 — note: returns 6 rows (end is INCLUSIVE) vs iloc[0:5] which returns 5

In [ ]:
df.loc[2, "price"]  # [row_label, col_name] — explicit column name beats positional index; survives column reordering

In [ ]:
# Most important pattern: boolean mask + column selection in one call
# This is the idiomatic pandas way to answer business questions like "show me high-value listings"
df.loc[df["price"] > 100, ["name", "price", "brand_name"]]

In [ ]:
# Multiple conditions use & (AND) / | (OR) — each condition MUST be in parentheses
# because & has higher precedence than comparison operators in Python
df.loc[(df["price"] > 20) & (df["shipping"] == 1)]

In [ ]:
# FIX: replaced fragile iloc index arithmetic with explicit column names
# Original used price_idx = df.columns.tolist().index("price") then iloc slicing —
# that breaks silently after feature engineering adds new columns and shifts position numbers
y = df["price"]                    # isolate the target variable by name, not by position
X = df.drop(columns=["price"])     # everything else becomes features — drop by name survives any column reordering

In [ ]:
# How many cheap unbranded listings exist? Answers whether "No Brand" items skew toward budget pricing
df.loc[(df["brand_name"].isnull()) & (df["price"] < 15)].shape[0]

---
## Data Cleaning
### Step 1 — Inspect Before Touching Anything

In [ ]:
df.isnull().sum()  # run BEFORE any cleaning so we know the baseline — gives us the numbers to justify every cleaning decision below

In [ ]:
df.isnull().sum() / len(df) * 100  # percentage view reveals that brand_name is ~43% null —
                                    # dropping it would lose nearly half the dataset, so filling is the only sane choice

In [ ]:
df.isnull().sum().sum()  # double .sum() collapses to a single scalar — quick go/no-go check before running expensive cleaning code

### Step 2 — Clean

In [ ]:
# category_name: only ~0.47% null — too sparse to fill meaningfully, dropping keeps the data honest
# without this drop, groupby("main_category") later would silently include a NaN bucket
df = df.dropna(subset=["category_name"])

In [ ]:
# brand_name: ~43% null — dropping would destroy nearly half the dataset
# "No Brand" acts as a legitimate category (unbranded items have distinct pricing patterns worth studying)
df["brand_name"] = df["brand_name"].fillna("No Brand")

In [ ]:
# Check for duplicates BEFORE dropping so we know whether the operation actually does anything
# (running drop_duplicates on zero dupes is harmless but knowing the count prevents false confidence)
df.duplicated().sum()

In [ ]:
# BUG FIX: original had df.drop_duplicates() with no assignment
# drop_duplicates() returns a NEW DataFrame — without assigning back, df is completely unchanged
df = df.drop_duplicates()

### Step 3 — Verify Cleaning Succeeded

In [ ]:
# BUG FIX: this was a MARKDOWN cell in the original — code in markdown never executes
# Running it now confirms all columns that were cleaned show 0 nulls
df.isnull().sum()  # category_name should be 0, brand_name should be 0

In [ ]:
df.shape  # row count after cleaning — confirms we only lost the ~0.47% null category rows, not a major cut

In [ ]:
# Confirm no row can simultaneously have null brand AND null category after our cleaning steps
# If this returns > 0, it means our cleaning logic had a gap we need to revisit
df[df["brand_name"].isnull() & df["category_name"].isnull()].shape[0]

---
## Feature Engineering — Creating Signals That Actually Help Models

In [ ]:
# BUG FIX: original had both sub_sub AND sub_sub_category columns (inconsistent naming)
# Standardized to sub_sub_category throughout so downstream groupby calls use the same name
#
# Splitting the slash-delimited category string into 3 tiers gives the model hierarchy signals:
# main_category (Women) is too broad | sub_sub_category (Shoulder Bag) is the most predictive for price
df["main_category"]     = df["category_name"].str.split("/").str[0]
df["sub_category"]      = df["category_name"].str.split("/").str[1]
df["sub_sub_category"]  = df["category_name"].str.split("/").str[2]

In [ ]:
# Normalize brand names so groupby/merge treats "Nike", "NIKE", " Nike " as the same brand
# Without this, a brand might appear as 3 separate groups, splitting its count and distorting avg price
df["brand_name"] = df["brand_name"].str.lower().str.strip()

In [ ]:
# BUG FIX: original checked x == "No description" — the actual Mercari placeholder is "No description yet"
# With the old string, this flag was always 1 (every item appeared "described"), making the feature useless
# Items with no description tend to sell for less — this binary flag captures that signal for the model
df["has_description"] = df["item_description"].apply(
    lambda x: 0 if x == "No description yet" else 1
)

In [ ]:
# Listing title length correlates with seller effort — detailed titles tend to attract more buyers
# and often command higher prices; gives the model a cheap proxy for listing quality
df["name_length"] = df["name"].str.len()

In [ ]:
# map() with a dict is ~10× faster than apply(lambda) for simple lookups on large Series
# Human-readable labels ("Good") make pivot tables and charts self-explanatory without a legend
condition_map = {1: "New", 2: "Like New", 3: "Good", 4: "Fair", 5: "Poor"}
df["condition_label"] = df["item_condition_id"].map(condition_map)

In [ ]:
# np.where is faster than apply() for nested conditionals on large Series (vectorized C loop)
# Budget/mid/premium tiers are standard Mercari market segments — grouping by these reveals different buyer behavior
df["price_bucket"] = df["price"].apply(
    lambda x: "budget" if x <= 10 else ("mid" if x <= 50 else "premium")
)

In [ ]:
# Isolate bundle listings to test whether multi-item sales command higher or lower unit prices
# na=False prevents crash when item name is NaN — without it, str.contains raises on null values
bundles = df[df["name"].str.lower().str.contains("bundle", na=False)]
bundles.shape[0]

In [ ]:
# Bundle avg price — if it's higher than overall avg, multi-item sellers are pricing strategically
print("Bundle avg:", bundles["price"].mean().round(2))

In [ ]:
# Overall avg for comparison baseline — the delta between these two numbers is the bundle premium/discount
print("Overall avg:", df["price"].mean().round(2))

---
## Groupby & Aggregation

### Basic Groupby

In [ ]:
# Mean price per category tells us WHICH categories attract premium sellers
# This drives Mercari's category-specific fee and marketing strategy
df.groupby("main_category")["price"].mean()

In [ ]:
# sort_values descending so Electronics (highest avg) jumps to the top —
# reading a sorted table takes 1 second vs scanning an alphabetical one for the max
df.groupby("main_category")["price"].mean().sort_values(ascending=False).round(2)

**`size()` vs `count()`** — `size()` counts ALL rows including nulls. `count()` counts only non-null values. Use `size()` when you want group volume; `count()` when you want valid data points.

In [ ]:
# size() gives total listings per category — reveals Women dominates (45%+ of all listings)
# critical for deciding where to focus seller acquisition campaigns
df.groupby("main_category").size()

In [ ]:
# count() on price gives the same result here because price has no nulls after cleaning
# use count() when the column you're studying might still have nulls — it tells you the "valid" sample size
df.groupby("main_category")["price"].count()

### Multi-Metric Aggregation with `.agg()`

In [ ]:
# BUG FIX: max_price was "mean" (copy-paste error) — in the old output, max_price equalled avg_price for every category
# Named aggregation (keyword=aggfunc) creates human-readable columns without a rename() call afterward
# This single table replaces four separate groupby calls
summary = df.groupby("main_category")["price"].agg(
    avg_price="mean",
    total_items="count",
    max_price="max",   # FIXED: was "mean" — now correctly shows the highest-priced item per category
    min_price="min"
).round(2).sort_values("avg_price", ascending=False)
summary

In [ ]:
# reset_index() demotes the groupby key from index to a regular column
# so the result is a plain DataFrame — downstream operations (sort, merge, plotting) work without surprises
result = df.groupby(["main_category", "shipping"])["price"].mean().round(2).reset_index()
result.sort_values("price", ascending=False).head(10)

### Practice — Brand Analysis

In [ ]:
# Exclude "no brand" sentinel so it doesn't pollute brand ranking results
# (we filled nulls with "no brand" earlier — useful for group analysis but irrelevant here)
branded = df[df["brand_name"] != "no brand"]

In [ ]:
# Top 3 by avg price — luxury brands (celine, canada goose) command 10–20× the typical listing price
# Mercari can use this to target premium seller acquisition for these brands
branded.groupby("brand_name")["price"].mean().sort_values(ascending=False).head(3).round(2)

In [ ]:
# Top 3 by listing volume — high volume ≠ high price (pink/nike/victoria's secret are mass market)
# Volume leaders are where Mercari should focus seller UX improvements
branded.groupby("brand_name").size().sort_values(ascending=False).head(3)

In [ ]:
# Interpretation: luxury brands = high avg price, low listing count (rare/expensive)
# Popular mass-market brands = high listing count, moderate price (volume-driven)
# These two segments require completely different seller incentive strategies

---
## Merge & Concat
### `concat` — Stack Same-Structure DataFrames Vertically

In [ ]:
# Simulates receiving data in two batches (e.g., morning/afternoon exports from Mercari API)
# Real-world: you'd read batch1.csv and batch2.csv; same concat logic applies
batch1 = df.iloc[:25000]
batch2 = df.iloc[25000:]

In [ ]:
# ignore_index=True resets the index to 0, 1, 2, 3... on the combined result
# Without it, the combined df has duplicate index values (0-24999 repeated twice),
# which causes silent bugs when using df.loc[n] — you'd get two rows instead of one
combined = pd.concat([batch1, batch2], ignore_index=True)

### `merge` — Join on a Key (Like SQL JOIN)

In [ ]:
# BUG FIX: original had "Woman" — the actual category name in df is "Women"
# That typo caused the entire Women group (22k+ rows = 45% of data) to produce NaN
# after the join, making market_segment and platform_fee useless for the Women category
cat_meta = pd.DataFrame({
    "main_category": ["Women", "Electronics", "Kids", "Beauty", "Men"],  # FIXED: was "Woman"
    "market_segment": ["Fashion", "Tech", "Family", "Personal Care", "Fashion"],
    "platform_fee":   [0.10, 0.08, 0.10, 0.12, 0.10]
})

In [ ]:
# Left join preserves ALL rows from df — categories with no meta entry get NaN
# (we deliberately only added 5 categories to cat_meta, so the other 5 will have NaN — expected)
df_enriched = pd.merge(df, cat_meta, on="main_category", how="left")

In [ ]:
# Row count must equal original df — a left join never drops rows
# If this shows MORE rows, there are duplicate keys in cat_meta (each df row joined to multiple meta rows)
df_enriched.shape

In [ ]:
# NaN count in market_segment tells us how many rows matched unregistered categories
# After the "Woman" bug fix, Women rows should now be populated; remaining NaNs are the 5 unmapped categories
df_enriched.isnull().sum()

In [ ]:
# Exact count of unmatched rows — should be significantly lower than 29k after the "Women" fix
# Remaining nulls are the 5 categories we intentionally omitted from cat_meta
df_enriched["market_segment"].isnull().sum()

---
## Pivot Tables — Spreadsheet-Style Cross-Tabulation

In [ ]:
# margins=True adds an "Overall" row/column — lets you compare a category's number to the fleet average
# fill_value=0 prevents NaN in cells with no data — cleaner for reporting than seeing blank cells
# This answers: "Does free shipping (shipping=0) correspond to higher listed prices across ALL categories?"
pd.pivot_table(
    df,
    values="price",
    index="main_category",
    columns="shipping",
    aggfunc="mean",
    fill_value=0,
    margins=True,
    margins_name="Overall"
).round(2)

In [ ]:
# Groupby produces the same numbers but as a flat multi-level index Series — harder to scan visually
# Pivot gives a 2D table — use pivot for reporting, groupby for programmatic downstream use
df.groupby(["main_category", "shipping"])["price"].mean()

In [ ]:
# Median instead of mean because price is right-skewed (a few $1500 bags inflate the mean)
# Median price per condition answers: "Do 'Poor' condition items actually sell cheaper across all categories?"
pd.pivot_table(
    df,
    values="price",
    index="main_category",
    columns="condition_label",
    aggfunc="median"
)

### Crosstab — Count-Based Cross-Tabulation

In [ ]:
# crosstab counts how many listings exist in each category × shipping combination
# use this BEFORE pivot_table when you want frequencies, not aggregated metrics
pd.crosstab(df["main_category"], df["shipping"])

In [ ]:
# normalize="index" converts raw counts to row-wise percentages (each row sums to 1.0)
# answers: "What FRACTION of each category's listings have seller-paid shipping?"
pd.crosstab(
    df["main_category"],
    df["shipping"],
    normalize="index"
).round(3)

In [ ]:
# Isolate the shipping=1 column so we can rank categories by free-shipping rate
# Handmade at 63% free shipping = sellers absorbing cost to compete on price perception
ct = pd.crosstab(df["main_category"], df["shipping"], normalize="index").round(3)
ct[1].sort_values(ascending=False)

---
## Sort & Rank

In [ ]:
# Sort descending to put the most expensive items first —
# sellers and pricing analysts scan from the top, so the interesting data should be at the top
df.sort_values("price", ascending=False)

In [ ]:
# Multi-key sort: category alphabetically, then within each category price descending
# This groups competitors together — a seller can instantly see where their price sits vs same-category peers
df.sort_values(["main_category", "price"], ascending=[True, False])

### `nlargest` / `nsmallest` — Faster Than `sort_values().head()`

In [ ]:
# nlargest uses a heap internally — O(n log k) vs O(n log n) for full sort
# For 50k rows picking top 10, it's noticeably faster and the intent is immediately readable
df.nlargest(10, "price")[["name", "price", "main_category"]]

In [ ]:
# Items priced at $0 are data quality flags — free listings either have missing price data
# or are placeholder posts; should be excluded before any pricing model is trained
df.nsmallest(5, "price")[["name", "price"]]

In [ ]:
# nlargest per group via groupby — shows the 5 priciest listings in EACH category
# more useful than a global top-10 which would be dominated entirely by Women's luxury bags
df.groupby("main_category")["price"].nlargest(5)

### Rank — Assign Ordinal Position to Each Row

In [ ]:
# method="dense" means tied prices get the SAME rank (no skipped numbers)
# so rank 1 = most expensive, and rank 2 = next unique price, not rank 1.5
# astype(int) strips the float decimals that rank() adds by default
df["price_rank"] = df["price"].rank(ascending=False, method="dense").astype(int)
df["price_rank"]

In [ ]:
# Within-category rank is a stronger ML feature than global rank —
# a $50 item ranks 1st in Kids but 200th in Electronics; the category context matters for price prediction
df["price_rank_in_cat"] = (
    df.groupby("main_category")["price"]
    .rank(ascending=False, method="dense")
    .astype(int)
)
df["price_rank_in_cat"]

In [ ]:
# pct=True normalises rank to [0, 1] — percentile is scale-invariant so it can be fed directly
# into models without further normalisation; 0.99 = top 1% most expensive items
df["price_percentile"] = df["price"].rank(pct=True).round(3)
df["price_percentile"]

In [ ]:
# Top 1% outliers — identifying which categories generate luxury-tier pricing
# Women + Electronics dominate the extremes, confirming they're the highest-value seller segments
top1 = df[df["price_percentile"] > 0.99]
print(top1.shape[0])
print(top1["main_category"].value_counts())

---
## NumPy on Mercari — Vectorized Math for ML Prep

### Log Transformation

In [ ]:
# Price is heavily right-skewed (mean=27, max=1506) — most ML models assume roughly normal distributions
# log1p instead of log because log(0) = -inf; log1p(0) = 0, which keeps zero-price items in the dataset
# This compresses the 0–1506 range down to 0–7.3, which gradient-based models learn from much faster
df["log_price"] = np.log1p(df["price"])

### IQR Outlier Detection

In [ ]:
# IQR method is robust to skewed data — unlike z-score which assumes normality
# The 1.5×IQR fence is the standard Tukey rule (the same one matplotlib uses for boxplot whiskers)
Q1 = np.percentile(df["price"], 25)
Q3 = np.percentile(df["price"], 75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR  # anything below this is statistically unusual (could be data entry error)
upper = Q3 + 1.5 * IQR  # anything above this is a premium outlier that could skew model training

print(f"Q1={Q1}, Q3={Q3}, IQR={IQR}")
print(f"Normal Range: {lower:.2f} to {upper:.2f}")

outliers = df[(df["price"] < lower) | (df["price"] > upper)]
print(f"Outliers: {outliers.shape[0]} ({outliers.shape[0]/len(df)*100:.2f}%)")

### `np.where()` — Vectorized If-Else (Faster Than `apply`)

In [ ]:
# np.where avoids a Python-level loop over 50k rows — runs at C speed via NumPy's broadcasting
# The is_outlier flag lets us filter outlier rows in one boolean mask without recomputing the bounds
df["is_price_outlier"] = np.where(
    (df["price"] < lower) | (df["price"] > upper), 1, 0
)

In [ ]:
# Nested np.where replicates the 3-tier bucketing logic from Feature Engineering —
# price_tier is used where we need a NEW column independent of the older price_bucket
# (e.g., to test whether tier definition affects model performance)
df["price_tier"] = np.where(
    df["price"] <= 10, "budget",
    np.where(df["price"] <= 50, "mid", "premium")
)

In [ ]:
# 90th percentile per category — more actionable than mean for pricing strategy:
# tells you the price ceiling a serious seller can realistically target in each category
p90 = (
    df.groupby("main_category")["price"]
    .apply(lambda x: np.percentile(x, 90))
    .round(2)
    .sort_values(ascending=False)
)
print(p90)

In [ ]:
# BUG FIX: original function had no return statement
# Without return, the function yields None for every group
# → apply() builds a Series of Nones → object dtype → .round() crashes with TypeError
#
# IQR per category reveals price SPREAD — Electronics has a huge spread (commodity to MacBook)
# vs Beauty which is tighter; spread informs how much price variance a model needs to explain
def get_iqr(x):
    return np.percentile(x, 75) - np.percentile(x, 25)  # FIXED: added return

iqr_per_cat = (
    df.groupby("main_category")["price"]
    .apply(get_iqr)
    .sort_values(ascending=False)
    .round(2)
)
iqr_per_cat

---
## Bug Fix Summary

| # | Location | Bug | Fix |
|---|---|---|---|
| 1 | Inspect nulls | `df.isnull().count()` always returns `len(df)` | Changed to `.isnull().sum()` |
| 2 | Remove duplicates | `df.drop_duplicates()` result was discarded | Assigned back: `df = df.drop_duplicates()` |
| 3 | `.agg()` summary | `max_price="mean"` copied from avg_price | Fixed to `max_price="max"` |
| 4 | `get_iqr` function | Missing `return` → `None` → object dtype → `.round()` crash | Added `return` statement |
| 5 | `has_description` flag | Checked `"No description"` (wrong string) → flag always 1 | Fixed to `"No description yet"` |
| 6 | `cat_meta` merge | `"Woman"` typo → 22k+ Women rows never matched | Fixed to `"Women"` |
| 7 | Verify section | Code was in a **markdown** cell — never executed | Converted to code cell |
| 8 | Data load | Absolute Windows path breaks on any other machine | Changed to relative path |
| 9 | Category columns | `sub_sub` and `sub_sub_category` both existed | Standardized to `sub_sub_category` |
| 10 | ML feature split | Fragile `iloc` index math breaks after feature engineering | Replaced with `df.drop(columns=["price"])` |